In [5]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-GGUF",
    filename="*Q4_K_M.gguf",
    n_ctx=24576,
    n_gpu_layers=-1,  # offload all layers to GPU
    verbose=False,
)
print("model loaded")

llama_context: n_ctx_seq (24576) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


model loaded


In [7]:
import json, time, os, csv, re
from datetime import datetime

DATA_PATH = "gold_standard_final_15.json"
os.makedirs("benchmark_results", exist_ok=True)

with open(DATA_PATH, "r") as f:
    papers = json.load(f)

In [8]:
PROMPT_TEMPLATE = """You are an expert biomedical researcher. Your task has TWO steps.

STEP 1 — STUDY QUALIFICATION:
Read the Methods section ONLY to determine: does this paper compare a disease group against a HEALTHY CONTROL group?

If YES → proceed to Step 2.
If NO → return this JSON and stop:
{{"qualified": false, "skip_reason": "brief explanation why", "disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}

STEP 2 — EXTRACTION:
Read ONLY the Abstract, Results, and Discussion sections (ignore Methods, figures, tables, supplementary materials).

Extract every microbe that is reported as significantly changed in the DISEASE group compared to HEALTHY CONTROLS. Be forgiving with microbe names (authors list them inconsistently) but be STRICT that there is a clear direction (increased or decreased). If direction is ambiguous, skip that microbe.

Return JSON:
{{"qualified": true, "skip_reason": null, "disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

Paper text:
{text}
JSON:"""

In [18]:
def smart_truncate(text):
    for marker in ["References\n", "REFERENCES\n", "Bibliography\n"]:
        idx = text.rfind(marker)
        if idx > 0:
            return text[:idx]
    return text

results = []

for i, paper in enumerate(papers):
    print(f"[{i+1}/{len(papers)}] {paper['title'][:70]}...")

    text = smart_truncate(paper["text"])
    prompt = PROMPT_TEMPLATE.format(text=text)
    start = time.time()

    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=4096,
        response_format={"type": "json_object"}
    )
    raw = output["choices"][0]["message"]["content"]
    elapsed = time.time() - start

    try:
        clean = raw
        if "</think>" in clean:
            clean = clean.split("</think>")[-1]
        clean = clean.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        parsed = json.loads(clean.strip())
    except:
        parsed = {"qualified": None, "skip_reason": "parse_error", "disease": paper["disease"], "taxa_enriched": [], "taxa_depleted": [], "parse_error": True}

    qualified = parsed.get("qualified", None)
    skip_reason = parsed.get("skip_reason", "")

    results.append({
        "title": paper["title"],
        "disease": paper["disease"],
        "in_gold_standard": paper["in_gold_standard"],
        "qualified": qualified,
        "skip_reason": skip_reason,
        "expected_enriched": paper.get("taxa_enriched", ""),
        "expected_depleted": paper.get("taxa_depleted", ""),
        "predicted_enriched": ", ".join(parsed.get("taxa_enriched", [])),
        "predicted_depleted": ", ".join(parsed.get("taxa_depleted", [])),
        "predicted_disease": parsed.get("disease", ""),
        "time_seconds": round(elapsed, 2),
        "parse_error": parsed.get("parse_error", False),
    })

    if not qualified:
        print(f"  SKIPPED: {skip_reason}")
    else:
        print(f"  enriched: {parsed.get('taxa_enriched', [])}")
        print(f"  depleted: {parsed.get('taxa_depleted', [])}")
    print(f"  {elapsed:.1f}s")

print(f"\nDone. {len(results)} papers.")

[1/15] Intestinal flora induces depression by mediating the dysregulation of ...
  enriched: ['g_Erysipelotrichaceae UCG-003', 'g_Erysipelotrichaceae UCG-014', 'g_pseudomonas']
  depleted: []
  14.0s
[2/15] Gut microbes exacerbate systemic inflammation and behavior disorders i...
  enriched: ['Porphyromonas', 'Akkermansia', 'Eubacterium siraeum', 'Akkermansia muciniphila', 'Fusobacterium varium', 'Megasphaera elsdenii', 'Clostridium aldenense', 'Bacteroides sp. OM05-12', 'Streptococcus sp. A12']
  depleted: ['Sutterella', 'Roseburia faecis', 'Eubacterium eligens', 'Sutterella parvirubra']
  24.5s
[3/15] Alterations in gut microbiota and metabolomic profiles in acute stroke...
  enriched: ['Faecalibacterium', 'Agathobacter', 'Dialister', 'Ruminococcus', 'Blautia', 'Barnesiella']
  depleted: ['Bacteroides', 'Escherichia-Shigella', 'Megamonas']
  16.9s
[4/15] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b...
  enriched: ['Collinsella', 'Desulfovibrio', 'Oscillospir

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
def parse_taxa(val):
    if not val or str(val) == "NaN" or str(val) == "nan":
        return []
    taxa = re.split(r'[,;]', str(val))
    cleaned = []
    for t in taxa:
        t = re.sub(r'\(.*?\)', '', t).strip().lower()
        t = re.sub(r'p\s*[<>=]\s*[\d.]+', '', t).strip()
        t = t.strip('.) ')
        if t and t != "nan" and len(t) > 2:
            cleaned.append(t)
    return cleaned

def match_taxa(predicted, expected):
    if not predicted and not expected:
        return [], []
    if not predicted:
        return [], expected
    if not expected:
        return [(p, "N/A (no expected)", 0.0) for p in predicted], []

    all_names = predicted + expected
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(all_names)
    sim = cosine_similarity(tfidf[:len(predicted)], tfidf[len(predicted):])

    matches = []
    matched_expected_idx = set()
    for i, pred in enumerate(predicted):
        best_j = sim[i].argmax()
        score = round(float(sim[i][best_j]), 3)
        matches.append((pred, expected[best_j], score))
        if score >= 0.5:
            matched_expected_idx.add(best_j)

    missed = [expected[j] for j in range(len(expected)) if j not in matched_expected_idx]
    return matches, missed
    

In [20]:
total_tp = total_fp = total_fn = 0
correct_skips = 0
wrong_skips = 0
wrong_extracts = 0

for r in results:
    print(f"\n{'='*60}")
    print(f"{r['title'][:70]}")
    print(f"Disease: {r['disease']} | Gold: {r['in_gold_standard']} | Qualified: {r['qualified']}")

    is_yes = r["in_gold_standard"] == "Yes"

    # track qualification accuracy
    if not r["qualified"]:
        if not is_yes:
            print(f"  ✅ Correctly skipped: {r['skip_reason']}")
            correct_skips += 1
        else:
            print(f"  ❌ WRONG SKIP (should have extracted): {r['skip_reason']}")
            wrong_skips += 1
            # count all expected as FN
            exp_enr = parse_taxa(r["expected_enriched"])
            exp_dep = parse_taxa(r["expected_depleted"])
            fn = len(exp_enr) + len(exp_dep)
            total_fn += fn
            print(f"  → TP=0  FP=0  FN={fn}")
        continue

    if not is_yes:
        pred_enr = parse_taxa(r["predicted_enriched"])
        pred_dep = parse_taxa(r["predicted_depleted"])
        n_pred = len(pred_enr) + len(pred_dep)
        if n_pred > 0:
            print(f"  ⚠️  Should have skipped, predicted {n_pred} taxa (not scored)")
            wrong_extracts += 1
        else:
            print(f"  ✅ Correctly abstained")
            correct_skips += 1
        continue

    # score Yes papers that were qualified
    pred_enr = parse_taxa(r["predicted_enriched"])
    pred_dep = parse_taxa(r["predicted_depleted"])
    exp_enr = parse_taxa(r["expected_enriched"])
    exp_dep = parse_taxa(r["expected_depleted"])

    enr_matches, enr_missed = match_taxa(pred_enr, exp_enr)
    dep_matches, dep_missed = match_taxa(pred_dep, exp_dep)

    if enr_matches:
        print("  ENRICHED:")
        for pred, exp, score in enr_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if enr_missed:
        print("  MISSED ENRICHED:")
        for m in enr_missed:
            print(f"    ⚠️  {m}")
    if dep_matches:
        print("  DEPLETED:")
        for pred, exp, score in dep_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if dep_missed:
        print("  MISSED DEPLETED:")
        for m in dep_missed:
            print(f"    ⚠️  {m}")

    tp = sum(1 for _, _, s in enr_matches + dep_matches if s >= 0.5)
    fp = sum(1 for _, _, s in enr_matches + dep_matches if s < 0.5)
    fn = len(enr_missed) + len(dep_missed)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    print(f"  → TP={tp}  FP={fp}  FN={fn}")

p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
rc = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1 = 2 * p * rc / (p + rc) if (p + rc) else 0

print(f"\n{'='*60}")
print(f"EXTRACTION (Yes papers only)")
print(f"TP={total_tp}  FP={total_fp}  FN={total_fn}")
print(f"Precision: {p:.3f}")
print(f"Recall:    {rc:.3f}")
print(f"F1:        {f1:.3f}")
print(f"\nQUALIFICATION")
print(f"Correct skips/abstains: {correct_skips}")
print(f"Wrong skips (missed Yes): {wrong_skips}")
print(f"Wrong extracts (missed No): {wrong_extracts}")


Intestinal flora induces depression by mediating the dysregulation of 
Disease: Stroke | Gold: No | Qualified: True
  ⚠️  Should have skipped, predicted 3 taxa (not scored)

Gut microbes exacerbate systemic inflammation and behavior disorders i
Disease: Other | Gold: Yes | Qualified: True
  ENRICHED:
    ✅ porphyromonas → porphyromonas  sim=1.0
    ✅ akkermansia → akkermansia  sim=1.0
    ✅ eubacterium siraeum → eubacterium siraeum  sim=1.0
    ✅ akkermansia muciniphila → akkermansia muciniphila  sim=1.0
    ✅ fusobacterium varium → fusobacterium varium  sim=1.0
    ✅ megasphaera elsdenii → megasphaera elsdenii  sim=1.0
    ✅ clostridium aldenense → clostridium aldenense  sim=1.0
    ✅ bacteroides sp. om05-12 → bacteroides sp. om05-12  sim=1.0
    ✅ streptococcus sp. a12 → streptococcus sp. a12  sim=1.0
  DEPLETED:
    ✅ sutterella → sutterella  sim=1.0
    ✅ roseburia faecis → roseburia faecis  sim=1.0
    ✅ eubacterium eligens → eubacterium eligens  sim=1.0
    ✅ sutterella parvirub